<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/07.%20Support%20Vector%20Machines%20and%20Kernel%20Methods/ch07_svm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 7: Support Vector Machines and Kernel Methods

## Intuisi: Maximum Margin Classifier

SVM mencari **hyperplane pemisah** yang memaksimalkan **margin** antara dua kelas. Hyperplane didefinisikan oleh:

$$\mathbf{w}^\top \mathbf{x} + b = 0$$

Margin geometris antara dua kelas adalah $\frac{2}{\|\mathbf{w}\|}$, sehingga memaksimalkan margin ekuivalen dengan **meminimalkan** $\|\mathbf{w}\|$.

### Hard-Margin SVM (Linearly Separable)

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 \quad \text{subject to} \quad y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1, \forall i$$

### Soft-Margin SVM (C-SVM)

Untuk data yang tidak sepenuhnya linearly separable, slack variables $\xi_i \geq 0$ diizinkan:

$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^n \xi_i$$

$C$ mengontrol tradeoff antara margin besar dan pelanggaran batasan:
- **C besar**: sedikit pelanggaran, margin kecil (cenderung overfit)
- **C kecil**: lebih banyak pelanggaran diizinkan, margin besar

### Kernel Trick

Untuk data yang tidak linearly separable, kernel memetakan data ke ruang dimensi tinggi tanpa perlu menghitung pemetaan secara eksplisit:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^\top \phi(\mathbf{x}_j)$$

| Kernel | Formula | Parameter |
|---|---|---|
| **Linear** | $K(x, x') = x^\top x'$ | - |
| **RBF (Gaussian)** | $K(x, x') = \exp(-\gamma\|x - x'\|^2)$ | $\gamma$ |
| **Polynomial** | $K(x, x') = (\gamma x^\top x' + r)^d$ | $\gamma, r, d$ |
| **Sigmoid** | $K(x, x') = \tanh(\gamma x^\top x' + r)$ | $\gamma, r$ |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_iris, make_classification, make_circles, make_moons
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
from matplotlib.colors import ListedColormap

cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

print(f'Breast Cancer: {X_cancer.shape}')
print(f'Iris: {X_iris.shape}')


## 7.1 SVM Decision Boundary Visualization

Visualisasi bagaimana SVM membentuk batas keputusan dan margin pada data 2D.


In [ ]:
# Visualize SVM decision boundary with different kernels
def plot_svm_boundary(ax, X, y, clf, title):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FFAAAA', '#AAFFAA']))
    ax.contour(xx, yy, Z, colors='black', linewidths=1)
    ax.scatter(X[y==0, 0], X[y==0, 1], c='red', s=30, edgecolors='k', linewidth=0.5, label='Class 0')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='green', s=30, edgecolors='k', linewidth=0.5, label='Class 1')
    # Highlight support vectors if available
    if hasattr(clf, 'support_vectors_'):
        sv = clf.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=100, facecolors='none', edgecolors='blue', linewidth=2, label='Support Vectors')
    ax.set_title(title)
    ax.legend(fontsize=7)

# Create linearly non-separable data
X_moons, y_moons = make_moons(n_samples=200, noise=0.15, random_state=42)
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for row, (X_data, y_data, dname) in enumerate([
    (X_moons, y_moons, 'Moons'),
    (X_circles, y_circles, 'Circles')
]):
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_data)

    kernels = [
        ('Linear', SVC(kernel='linear', C=1.0)),
        ('RBF C=1', SVC(kernel='rbf', C=1.0, gamma='scale')),
        ('RBF C=10', SVC(kernel='rbf', C=10.0, gamma='scale')),
        ('Poly d=3', SVC(kernel='poly', C=1.0, degree=3, gamma='scale'))
    ]
    for ax, (kname, clf) in zip(axes[row], kernels):
        clf.fit(X_sc, y_data)
        acc = clf.score(X_sc, y_data)
        plot_svm_boundary(ax, X_sc, y_data, clf, f'{dname}: {kname}\nTrain acc={acc:.3f}')

plt.tight_layout()
plt.show()


## 7.2 SVC untuk Klasifikasi (Breast Cancer)

Hyperparameter tuning dengan GridSearchCV untuk menemukan kombinasi kernel, C, dan gamma terbaik.


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cancer, y_cancer, test_size=0.25, random_state=42, stratify=y_cancer
)

# Compare kernels
kernels = [
    ('Linear', SVC(kernel='linear', C=1.0, random_state=42)),
    ('RBF C=1', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)),
    ('RBF C=10', SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42)),
    ('Poly d=2', SVC(kernel='poly', degree=2, C=1.0, gamma='scale', random_state=42)),
    ('Poly d=3', SVC(kernel='poly', degree=3, C=1.0, gamma='scale', random_state=42)),
]

print('=== SVM Kernel Comparison (Breast Cancer) ===')
print(f'{"Kernel":<20} {"Test Acc":>10} {"5-CV Acc":>10}')
print('-' * 42)
for kname, svc in kernels:
    pipe = Pipeline([('scaler', StandardScaler()), ('svc', svc)])
    pipe.fit(X_tr, y_tr)
    test_acc = pipe.score(X_te, y_te)
    cv_acc = cross_val_score(pipe, X_cancer, y_cancer, cv=5).mean()
    print(f'{kname:<20} {test_acc:>10.4f} {cv_acc:>10.4f}')


In [ ]:
# Grid Search for best hyperparameters
param_grid = {
    'svc__C': [0.1, 1.0, 10.0, 100.0],
    'svc__gamma': ['scale', 'auto', 0.001, 0.01],
    'svc__kernel': ['rbf']
}

pipe_grid = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42))
])

grid_search = GridSearchCV(pipe_grid, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_cancer, y_cancer)

print('=== Grid Search Results ===')
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV accuracy: {grid_search.best_score_:.4f}')

best_svm = grid_search.best_estimator_
best_svm.fit(X_tr, y_tr)
print(f'Test accuracy: {best_svm.score(X_te, y_te):.4f}')
print(classification_report(y_te, best_svm.predict(X_te), target_names=cancer.target_names))


## 7.3 SVR: Support Vector Regression

SVR menggunakan konsep margin toleransi $\varepsilon$ (epsilon-tube). Prediksi yang masuk dalam tube tidak menghasilkan error:

$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_i (\xi_i + \xi_i^*)$$

$$\text{subject to} \quad |y_i - (\mathbf{w}^\top \mathbf{x}_i + b)| \leq \varepsilon + \xi_i$$


In [ ]:
from sklearn.svm import SVR
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
Xd, yd = diabetes.data, diabetes.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=42)

# Compare SVR kernels
svr_models = [
    ('SVR Linear', SVR(kernel='linear', C=100)),
    ('SVR RBF', SVR(kernel='rbf', C=100, gamma='scale')),
    ('SVR Poly d=2', SVR(kernel='poly', C=100, degree=2, gamma='scale')),
]

print('=== SVR Comparison (Diabetes Dataset) ===')
from sklearn.linear_model import LinearRegression
scaler_d = StandardScaler()
Xd_tr_sc = scaler_d.fit_transform(Xd_tr)
Xd_te_sc = scaler_d.transform(Xd_te)

for name, svr in svr_models:
    svr.fit(Xd_tr_sc, yd_tr)
    yd_pred = svr.predict(Xd_te_sc)
    rmse = np.sqrt(mean_squared_error(yd_te, yd_pred))
    r2 = r2_score(yd_te, yd_pred)
    print(f'{name:<18}: RMSE={rmse:.2f}, R2={r2:.4f}, SVs={svr.support_vectors_.shape[0]}')

# Best SVR visualization
best_svr = SVR(kernel='rbf', C=100, gamma='scale', epsilon=5)
best_svr.fit(Xd_tr_sc, yd_tr)
yd_pred_best = best_svr.predict(Xd_te_sc)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(yd_te, yd_pred_best, alpha=0.5, s=30, color='steelblue')
ax.plot([yd_te.min(), yd_te.max()], [yd_te.min(), yd_te.max()], 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Actual values')
ax.set_ylabel('Predicted values')
ax.set_title(f'SVR (RBF): Actual vs Predicted - Diabetes Dataset\nRMSE={np.sqrt(mean_squared_error(yd_te, yd_pred_best)):.2f}, R2={r2_score(yd_te, yd_pred_best):.4f}')
ax.legend()
plt.tight_layout()
plt.show()


## Ringkasan: Key Takeaways Chapter 7

| Konsep | Poin Kunci |
|---|---|
| **Margin Maximization** | SVM mencari hyperplane dengan margin terbesar; ini membuatnya robust terhadap noise |
| **Support Vectors** | Hanya titik-titik di atau di luar margin yang mendefinisikan hyperplane; model sparse |
| **Parameter C** | C kecil = margin besar, lebih toleran (bias tinggi); C besar = margin kecil, ketat (variance tinggi) |
| **Kernel Trick** | Memungkinkan pemisahan non-linear tanpa perlu menghitung pemetaan eksplisit ke dimensi tinggi |
| **RBF Kernel** | Paling umum digunakan; gamma mengontrol "bandwidth" → gamma besar = decision boundary kompleks |
| **SVR epsilon** | Prediksi yang masuk dalam epsilon-tube tidak memberikan kontribusi pada loss |
| **Normalisasi** | Wajib! SVM sangat sensitif terhadap skala fitur |

### Panduan Pemilihan Kernel
- **Data linear separable**: Linear kernel → paling interpretable
- **Data non-linear, fitur sedikit**: RBF kernel → paling fleksibel
- **NLP/text data**: Linear kernel → fitur sudah high-dimensional
- **Structured data**: Mulai dengan RBF, lalu tuning C dan gamma
